# Deployment

## Review

We built up to an agent with memory:

- `act` - let the model call specific tools
- `observe` - pass the tool output back to the model
- `reason` - let the model reason about the tool output to decide what to do next (e.g., call another tool or just respond directly)
- `persist state` - use an in memory checkpointer to support long-running conversations with interruptions

## Goals

Now, we'll cover how to actually deploy our agent locally to Studio and to `LangSmith Deployment`.


## Concepts

There are a few central concepts to understand:

`LangGraph`

- Python and JavaScript library
- Allows creation of agent workflows

`LangGraph API`

- Bundles the graph code
- Provides a task queue for managing asynchronous operations
- Offers persistence for maintaining state across interactions

`LangSmith Deployment`

- Hosted service for the LangGraph API
- Allows deployment of graphs from GitHub repositories
- Also provides monitoring and tracing for deployed graphs
- Accessible via a unique URL for each deployment

`LangSmith Studio`

- Integrated Development Environment (IDE) for LangGraph applications
- Uses the API as its back-end, allowing real-time testing and exploration of graphs
- Can be run locally or with cloud-deployment. See below.

`LangGraph SDK`

- Python library for programmatically interacting with LangGraph graphs
- Provides a consistent interface for working with graphs, whether served locally or in the cloud
- Allows creation of clients, access to assistants, thread management, and execution of runs


## Testing Locally

### Studio

To start the local development server, run the following command in your terminal in the `/studio` directory in this module:

```
langgraph dev
```

You should see the following output:

```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

Open your browser and navigate to the **Studio UI** URL shown above.


In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langgraph_sdk import get_client

In [ ]:
# This is the URL of the local development server
URL = "http://127.0.0.1:2024"
client = get_client(url=URL)

# Search all hosted graphs
assistants = await client.assistants.search()

In [ ]:
assistants[-3]

In [ ]:
# We create a thread for tracking the state of our run
thread = await client.threads.create()

Now, we can run our agent [with `client.runs.stream`](https://docs.langchain.com/oss/python/langgraph/graph-api/#stream-and-astream) with:

- The `thread_id`
- The `graph_id`
- The `input`
- The `stream_mode`

We'll discuss streaming in depth in a future module.

For now, just recognize that we are [streaming](https://docs.langchain.com/langsmith/streaming) the full value of the state after each step of the graph with `stream_mode="values"`.

The state is captured in the `chunk.data`.


In [ ]:
from langchain.messages import HumanMessage

# Input
input = {"messages": [HumanMessage(content="Multiply 3 by 2.")]}

# Stream
async for chunk in client.runs.stream(
    thread['thread_id'],
    "agent",
    input=input,
    stream_mode="values",
):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])

## Testing with Cloud

We can deploy to Cloud via LangSmith, as outlined [here](https://docs.langchain.com/langsmith/deployment-quickstart#deploy-from-github-with-langgraph-cloud).

### Create a New Repository on GitHub

- Go to your GitHub account
- Click on the "+" icon in the upper-right corner and select `"New repository"`
- Name your repository (e.g., `inroduction-to-langgraph`)

### Add Your GitHub Repository as a Remote

- Go back to your terminal where you cloned `inroduction-to-langgraph` at the start of this course
- Add your newly created GitHub repository as a remote

```
git remote add origin https://github.com/your-username/your-repo-name.git
```

- Push to it

```
git push -u origin main
```

### Connect LangSmith to your GitHub Repository

- Go to [LangSmith](hhttps://smith.langchain.com/)
- Click on `deployments` tab on the left LangSmith panel
- Add `+ New Deployment`
- Then, select the Github repository (e.g., `inroduction-to-langgraph`) that you just created for the course
- Point the `LangGraph API config file` at one of the `studio` directories
- For example, for module-1 use: `module-1/studio/langgraph.json`
- Set your API keys (e.g., you can just copy from your `module-1/studio/.env` file)

### Work with your deployment

We can then interact with our deployment a few different ways:

- With the SDK, as before.
- With [LangSmith Studio](https://docs.langchain.com/langsmith/deployment-quickstart#3-test-your-application-in-studio).

To use the SDK here in the notebook, simply ensure that `LANGSMITH_API_KEY` is set!


In [ ]:
# Replace this with the URL of your deployed graph
URL = "https://inroduction-to-langgraph-12345.default.us.langgraph.app"
client = get_client(url=URL)

# Search all hosted graphs
assistants = await client.assistants.search()

In [ ]:
# Select the agent
agent = assistants[0]

In [ ]:
agent

In [ ]:
from langchain.messages import HumanMessage

# We create a thread for tracking the state of our run
thread = await client.threads.create()

# Input
input = {"messages": [HumanMessage(content="Multiply 3 by 2.")]}

# Stream
async for chunk in client.runs.stream(
    thread['thread_id'],
    "agent",
    input=input,
    stream_mode="values",
):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])